# Notebook 02c: Phân tích Chuyên sâu Câu hỏi Q3 & Q4 (Drug ROR & Off-label Use)
**Đề tài:** Phân tích Sự cố Bất lợi Thuốc Thú y (OpenFDA Animal & Veterinary Adverse Events)

**Môn học:** CSC17104 – Lập trình cho Khoa học Dữ liệu  
**Thành viên thực hiện:** Huỳnh Tuấn Kiệt (M3) — Nhánh Git: `feature/kiet`  
**Phạm vi sở hữu:** **Câu hỏi Q3** & **Câu hỏi Q4** (Non-ML Questions)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_processed
from src.preprocessing import log_row_count_change
from src.visualization import set_project_plot_style
set_project_plot_style()

In [ ]:
# Tải dữ liệu đã qua tiền xử lý chuẩn hóa từ data/processed/
data = load_processed(PROJECT_ROOT / 'data' / 'processed')
reports = data['reports']
drugs = data['drugs']
reactions = data['reactions']
outcomes = data['outcomes']

print(f"[Data Loaded] Reports: {reports.shape}, Drugs: {drugs.shape}, Reactions: {reactions.shape}, Outcomes: {outcomes.shape}")

---
## 1. Câu hỏi Q3: Tỷ suất Đồng xuất hiện (ROR) Hoạt chất x Triệu chứng (Safety Issue vs Lack of Efficacy)

### Phát biểu câu hỏi & Động lực (Question & Motivation)
- **Phát biểu câu hỏi:**  
  > **"Các hoạt chất (`active_ingredients`) và đường dùng (`route`) nào thường đi kèm với những reaction cụ thể, và cấu trúc đồng xuất hiện (co-occurrence pattern / Reporting Odds Ratio — ROR) này khác nhau như thế nào giữa nhóm Safety Issue và Lack Of Expected Effectiveness trong năm 2025?"**
- **Người quan tâm & Quyết định hỗ trợ (Target Users & Decisions):**
  - **Chuyên gia Cảnh giác dược (FDA CVM) & Bộ phận R&D Dược phẩm:** Cần phân biệt rõ hoạt chất nào gây tín hiệu độc tính sinh học thực sự (Safety Issue — nôn mửa, co giật, tử vong) với hoạt chất nào đang gặp tình trạng giảm hiệu lực/kháng thuốc (Lack of Expected Effectiveness — ví dụ thuốc trị ve, rận, bọ chét nhưng không diệt được ký sinh trùng).
- **Tiêu chí độ sâu áp dụng (Mục 3.3):**
  - `D1` (Combining data): Nối 3 bảng `reports`, `drugs`, `reactions` qua `unique_aer_id_number` (kiểm tra toàn vẹn khóa nối, chống bùng nổ Cartesian).
  - `D2` (Reshaping & time): Reshape ma trận đồng xuất hiện giữa các cặp (Hoạt chất, Triệu chứng) qua pivot table và melt.
  - `D4` (Controlling for a confounder): Phân tầng phân tích theo loài động vật chủ đạo (Chó vs Mèo) để tránh tín hiệu giả do loài gây ra.
  - `D5` (Quantifying uncertainty): Tính toán Reporting Odds Ratio (ROR) từ bảng liên hợp $2 \times 2$ cùng khoảng tin cậy 95% Confidence Interval theo chuẩn dịch tễ dược.

### 1.1. Tiền xử lý cho Q3 (Preprocessing & Row Count Tracking)
*(Lọc dữ liệu năm 2025, chuẩn hóa tên hoạt chất, phân tách Safety Issue vs Lack of Efficacy, theo dõi số dòng)*

In [ ]:
# TODO [Kiệt - M3]: Triển khai tiền xử lý cho Q3 kèm log_row_count_change

### 1.2. Phương pháp phân tích (Analysis Method)
*(Xây dựng bảng liên hợp 2x2, công thức ROR = (a*d)/(b*c), tính SE(ln ROR) và khoảng tin cậy 95%)*

In [ ]:
# TODO [Kiệt - M3]: Triển khai tính toán ROR, 95% CI và phân tích khác biệt Safety vs Efficacy

### 1.3. Kết quả & Trực quan hóa (Results & Visualizations)
*(Vẽ tối thiểu 2 biểu đồ: Forest plot ROR với 95% CI cho Top hoạt chất; Heatmap so sánh tín hiệu Safety vs Lack of Efficacy)*

In [ ]:
# TODO [Kiệt - M3]: Vẽ >= 2 biểu đồ chuẩn mực cho Q3

### 1.4. Trả lời Câu hỏi & Thảo luận (Answer & Practical Meaning)
*(Trả lời bằng các cặp Hoạt chất - Triệu chứng có ROR cao nhất vượt ngưỡng tín hiệu (ROR > 2 và cận dưới CI > 1), ý nghĩa cảnh giác dược)*

---
## 2. Câu hỏi Q4: Sử dụng Thuốc Sai Nhãn (Off-label), Đa Trị Liệu & Mức độ Nghiêm Trọng / Kết Cục

### Phát biểu câu hỏi & Động lực (Question & Motivation)
- **Phát biểu câu hỏi:**  
  > **"Việc dùng thuốc đúng nhãn (`used_according_to_label = true`) so với dùng sai nhãn (`off-label`) trong năm 2025 có liên hệ như thế nào với mức độ nghiêm trọng (`serious_ae`) và kết cục y khoa (`outcome`) sau khi kiểm soát biến nhiễu là loài vật nuôi và đối tượng báo cáo (`primary_reporter`)?"**
- **Người quan tâm & Quyết định hỗ trợ (Target Users & Decisions):**
  - **Hiệp hội Bác sĩ Thú y & Cơ quan Quản lý Dược:** Đánh giá rủi ro an toàn thực tế khi sử dụng thuốc ngoài chỉ định (extra-label/off-label) để ban hành các phác đồ khuyến cáo thận trọng hoặc siết chặt quy chế kê đơn.
- **Tiêu chí độ sâu áp dụng (Mục 3.3):**
  - `D1` (Combining data): Kết hợp 3 bảng `reports`, `drugs`, `outcomes` qua `unique_aer_id_number`.
  - `D3` (Derived variables): Tạo các biến dẫn xuất cấp báo cáo `has_off_label_drug` (có ít nhất 1 thuốc dùng sai nhãn), `is_polypharmacy` (dùng $\ge 2$ thuốc), và biến thứ bậc kết cục xấu nhất `worst_outcome_rank` (Tử vong / An tử > Chưa hồi phục > Đã hồi phục).
  - `D4` (Controlling for a confounder): Kiểm soát đồng thời biến loài (`species`) và đối tượng báo cáo (`primary_reporter`: Bác sĩ thú y vs Khách hàng) bằng hồi quy đa biến Logistic/Ordinal Logistic.
  - `D6` (Robustness check): Kiểm tra độ vững của kết luận khi xem xét trường hợp nhóm khuyết thông tin nhãn (`missing / unknown`) được quy về dùng đúng nhãn so với khi loại bỏ hoàn toàn.

### 2.1. Tiền xử lý cho Q4 (Preprocessing & Row Count Tracking)
*(Tổng hợp biến cấp ca báo cáo từ drugs và outcomes, xử lý nhãn, log row count change)*

In [ ]:
# TODO [Kiệt - M3]: Triển khai tiền xử lý cho Q4 kèm log_row_count_change

### 2.2. Phương pháp phân tích (Analysis Method)
*(Phân tầng Mantel-Haenszel, mô hình hồi quy đa biến kiểm soát nhiễu species và primary_reporter)*

In [ ]:
# TODO [Kiệt - M3]: Triển khai phân tích Q4 (Odds Ratio, kiểm soát biến nhiễu D4, Robustness D6)

### 2.3. Kết quả & Trực quan hóa (Results & Visualizations)
*(Vẽ tối thiểu 2 biểu đồ: Tỷ lệ biến cố nghiêm trọng giữa Đúng nhãn vs Sai nhãn theo đối tượng báo cáo; Phân bố kết cục y khoa worst_outcome_rank)*

In [ ]:
# TODO [Kiệt - M3]: Vẽ >= 2 biểu đồ chuẩn mực cho Q4

### 2.4. Trả lời Câu hỏi & Thảo luận (Answer & Practical Meaning)
*(Trả lời bằng số liệu chênh lệch rủi ro, ý nghĩa của việc kiểm soát người báo cáo, rủi ro off-label thực tế)*